# MULCURSIO: Analyze Session Data

This notebook provides tools for exploring and analyzing backroom conversation session data.

## Setup

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent directories to path
sys.path.insert(0, str(Path().absolute().parent))
sys.path.insert(0, str(Path().absolute().parent.parent))

# Set plotting style
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Select Session to Analyze

In [ ]:
# List available sessions
sessions_dir = Path("../sessions")
sessions = sorted([d for d in sessions_dir.iterdir() if d.is_dir()])

print(f"Available sessions ({len(sessions)}):")
for i, session in enumerate(sessions):
    print(f"  [{i}] {session.name}")

# Select session (change index as needed)
session_idx = -1  # -1 = most recent
session_dir = sessions[session_idx]
print(f"\nAnalyzing: {session_dir.name}")

## 2. Load Session Manifest

In [ ]:
# Load manifest
manifest_path = session_dir / "manifest.json"
with open(manifest_path, 'r') as f:
    manifest = json.load(f)

print(f"Session: {manifest['session_id']}")
print(f"Created: {manifest['created_at']}")
print(f"Total turns: {manifest['total_turns']}")
print(f"\nConfig:")
for key, value in manifest['config'].items():
    print(f"  {key}: {value}")

## 3. Load Vocabulary

In [ ]:
# Load vocab
vocab_path = session_dir / "vocab.json"
with open(vocab_path, 'r') as f:
    vocab = json.load(f)

print(f"Vocabulary size: {len(vocab)}")

# Helper function to decode token IDs
def decode_token(token_id):
    return vocab.get(str(token_id), "<UNK>")

## 4. Analyze Entropy Across Turns

Let's look at how entropy evolves over the conversation.

In [ ]:
# Collect entropy statistics for each turn
entropy_stats = []

for turn_info in manifest['turns']:
    turn_id = turn_info['turn_id']
    agent = turn_info['agent']
    metrics_file = session_dir / turn_info['metrics_file']
    
    if metrics_file.exists():
        metrics = np.load(metrics_file)
        entropy = metrics['entropy']
        
        entropy_stats.append({
            'turn': turn_id,
            'agent': agent,
            'mean_entropy': entropy.mean(),
            'max_entropy': entropy.max(),
            'min_entropy': entropy.min(),
            'std_entropy': entropy.std()
        })

# Plot entropy over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Separate by agent
agent_a_stats = [s for s in entropy_stats if s['agent'] == 'A']
agent_b_stats = [s for s in entropy_stats if s['agent'] == 'B']

# Plot mean entropy
ax1.plot([s['turn'] for s in agent_a_stats], [s['mean_entropy'] for s in agent_a_stats], 
         'o-', label='Agent A (baseline)', alpha=0.7)
ax1.plot([s['turn'] for s in agent_b_stats], [s['mean_entropy'] for s in agent_b_stats], 
         's-', label='Agent B (divination)', alpha=0.7)
ax1.set_xlabel('Turn')
ax1.set_ylabel('Mean Entropy')
ax1.set_title('Mean Token Entropy Over Conversation')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot entropy range
ax2.plot([s['turn'] for s in agent_a_stats], [s['std_entropy'] for s in agent_a_stats], 
         'o-', label='Agent A (baseline)', alpha=0.7)
ax2.plot([s['turn'] for s in agent_b_stats], [s['std_entropy'] for s in agent_b_stats], 
         's-', label='Agent B (divination)', alpha=0.7)
ax2.set_xlabel('Turn')
ax2.set_ylabel('Std Dev of Entropy')
ax2.set_title('Entropy Variability Over Conversation')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nOverall statistics:")
print(f"Agent A mean entropy: {np.mean([s['mean_entropy'] for s in agent_a_stats]):.3f}")
print(f"Agent B mean entropy: {np.mean([s['mean_entropy'] for s in agent_b_stats]):.3f}")

## 5. Analyze Token Probability

How confident is each agent in their token choices?

In [ ]:
# Collect probability statistics
prob_stats = []

for turn_info in manifest['turns']:
    turn_id = turn_info['turn_id']
    agent = turn_info['agent']
    metrics_file = session_dir / turn_info['metrics_file']
    
    if metrics_file.exists():
        metrics = np.load(metrics_file)
        chosen_prob = metrics['chosen_prob']
        
        prob_stats.append({
            'turn': turn_id,
            'agent': agent,
            'mean_prob': chosen_prob.mean(),
            'min_prob': chosen_prob.min(),
            'tokens_below_50pct': (chosen_prob < 0.5).sum() / len(chosen_prob)
        })

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

agent_a_probs = [s for s in prob_stats if s['agent'] == 'A']
agent_b_probs = [s for s in prob_stats if s['agent'] == 'B']

ax.plot([s['turn'] for s in agent_a_probs], [s['mean_prob'] for s in agent_a_probs], 
        'o-', label='Agent A (baseline)', alpha=0.7)
ax.plot([s['turn'] for s in agent_b_probs], [s['mean_prob'] for s in agent_b_probs], 
        's-', label='Agent B (divination)', alpha=0.7)
ax.axhline(y=0.5, color='r', linestyle='--', alpha=0.3, label='50% confidence')
ax.set_xlabel('Turn')
ax.set_ylabel('Mean Probability of Chosen Token')
ax.set_title('Model Confidence Over Conversation')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nConfidence statistics:")
print(f"Agent A mean confidence: {np.mean([s['mean_prob'] for s in agent_a_probs]):.3f}")
print(f"Agent B mean confidence: {np.mean([s['mean_prob'] for s in agent_b_probs]):.3f}")

## 6. Inspect Specific Turn in Detail

In [ ]:
# Select a turn to inspect
turn_id = 0  # Change this to inspect different turns

turn_info = manifest['turns'][turn_id]
metrics_file = session_dir / turn_info['metrics_file']

print(f"Turn {turn_id} | Agent {turn_info['agent']}")
print(f"Control coefficient: {turn_info['control_coeff']}")
print(f"\nPrompt: {turn_info['prompt'][:200]}...")
print(f"\nResponse: {turn_info['response'][:500]}...")
print("\n" + "="*60)

# Load metrics
metrics = np.load(metrics_file)

# Decode tokens
output_ids = metrics['output_ids']
tokens = [decode_token(int(tid)) for tid in output_ids]

# Show token-level details for first 20 tokens
print("\nToken-level analysis (first 20 tokens):")
print(f"{'Token':<20} {'Prob':<10} {'Entropy':<10} {'Rank':<10}")
print("-" * 60)

for i in range(min(20, len(tokens))):
    token = tokens[i]
    prob = metrics['chosen_prob'][i]
    entropy = metrics['entropy'][i]
    rank = metrics['chosen_rank'][i]
    
    print(f"{token:<20} {prob:<10.4f} {entropy:<10.3f} {rank:<10}")

## 7. Visualize Top-K Alternatives

For a specific token, what were the alternative possibilities?

In [ ]:
# Select a token position to inspect
token_idx = 5  # Change this to inspect different tokens

if token_idx < len(output_ids):
    chosen_token = decode_token(int(output_ids[token_idx]))
    top_k_tokens = metrics['top_k_tokens'][token_idx]
    top_k_probs = metrics['top_k_probs'][token_idx]
    
    print(f"Token {token_idx}: '{chosen_token}'")
    print(f"\nTop-10 alternatives:")
    print(f"{'Rank':<6} {'Token':<20} {'Probability':<12}")
    print("-" * 50)
    
    for rank, (tid, prob) in enumerate(zip(top_k_tokens, top_k_probs)):
        token = decode_token(int(tid))
        marker = " ← CHOSEN" if token == chosen_token else ""
        print(f"{rank+1:<6} {token:<20} {prob:<12.6f} {marker}")
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    tokens_decoded = [decode_token(int(tid)) for tid in top_k_tokens]
    colors = ['red' if t == chosen_token else 'steelblue' for t in tokens_decoded]
    
    ax.bar(range(len(top_k_probs)), top_k_probs, color=colors, alpha=0.7)
    ax.set_xticks(range(len(tokens_decoded)))
    ax.set_xticklabels(tokens_decoded, rotation=45, ha='right')
    ax.set_ylabel('Probability')
    ax.set_title(f'Top-10 Token Alternatives (Position {token_idx})')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

## 8. Export Highlight Moments

Find and export the most interesting moments (high entropy, low confidence, etc.)

In [ ]:
# Find high-entropy turns
high_entropy_turns = sorted(entropy_stats, key=lambda x: x['mean_entropy'], reverse=True)[:5]

print("Top 5 highest-entropy turns:")
for i, stats in enumerate(high_entropy_turns):
    turn_info = manifest['turns'][stats['turn']]
    print(f"\n{i+1}. Turn {stats['turn']} (Agent {stats['agent']})")
    print(f"   Mean entropy: {stats['mean_entropy']:.3f}")
    print(f"   Response: {turn_info['response'][:200]}...")

# Find low-confidence turns
low_confidence_turns = sorted(prob_stats, key=lambda x: x['mean_prob'])[:5]

print("\n" + "="*60)
print("\nTop 5 lowest-confidence turns:")
for i, stats in enumerate(low_confidence_turns):
    turn_info = manifest['turns'][stats['turn']]
    print(f"\n{i+1}. Turn {stats['turn']} (Agent {stats['agent']})")
    print(f"   Mean confidence: {stats['mean_prob']:.3f}")
    print(f"   Response: {turn_info['response'][:200]}...")